# Neighbour Similarity Thematic Evolution

This notebook traces thematic evolution in ADHD and autism discourse with the pair-wise neighbour-similarity method used by [Vylomova and Haslam (2021)](https://langsci-press.org/catalog/view/303/3028/2375-1) and adapted as Neighbours Similarity Evolution by [Iacob and Uban (2026)](https://aclanthology.org/2026.lchange-1.12/). The analysis trains diachronic type-level Word2Vec spaces and asks which content words become more or less distributionally close to the canonical target concept over publication years.

Thematic evolution is not treated as another scalar SIBling index. It is an interpretive layer: the main objects are neighbour trajectories and the annual top-neighbour tables that explain what kinds of contexts surround each target concept over time.

## Setup

The analysis uses the shared LSC mention-context table and the locked frame-classifier handoff. ADHD and autism raw forms are canonicalised to one concept token before Word2Vec training so that the model estimates concept-level neighbour trajectories rather than spelling- or acronym-specific trajectories. Baseline terms are not included because this stage is qualitative and target-frame specific.

The processed output folder is deliberately narrow: it keeps the full annual top-neighbour table, the stable plotted-neighbour table, the plotted trajectories, and a small execution summary. Input, token, model, training, and audit diagnostics are displayed in the notebook rather than exported as separate handoff files.


In [1]:

from __future__ import annotations

import copy
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from scipy import stats
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
INTERIM_DIR = PROJECT_ROOT / "data/interim/lsc/thematic_evolution"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/lsc/thematic_evolution"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/thematic_evolution"
APPENDIX_FIGURE_DIR = FIGURE_DIR / "appendix"
MODEL_DIR = INTERIM_DIR / "word2vec_models"

for directory in [INTERIM_DIR, PROCESSED_DIR, FIGURE_DIR, APPENDIX_FIGURE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
MODELED_FRAME_STRATA = ["substantive_core_overall", "clinical_only", "lived_only"]
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder",
    "lived_only": "Lived experience",
    "mixed": "Mixed",
}
TARGET_CONCEPT_TOKEN = {"ADHD": "adhd_concept", "Autism": "autism_concept"}

WORD2VEC_PARAMS = {
    "vector_size": 200,
    "window": 10,
    "min_count": 5,
    "sg": 1,
    "workers": 1,
    "seed": 13,
    "global_epochs": 10,
    "year_epochs": 10,
}
REBUILD_TOKENISED_CONTEXTS = False
REBUILD_WORD2VEC_MODELS = False
TOKENISATION_CONFIG_VERSION = 1
MIN_CONTEXT_TOKENS = 5
MIN_TARGET_YEAR_TOKEN_COUNT = 20
MIN_NEIGHBOUR_YEAR_TOKEN_COUNT = 5
ANNUAL_TOP_N = 5
PLOTTED_NEIGHBOURS = 5
MIN_PLOTTED_TOP5_YEARS = 2
MIN_PLOTTED_FINITE_YEARS = 10

RETAINED_PROCESSED_OUTPUTS = {
    "lsc_thematic_annual_top_neighbours.csv",
    "lsc_thematic_plotted_neighbours.csv",
    "lsc_thematic_neighbour_similarity_trajectories.csv",
    "lsc_thematic_execution_summary.json",
}
for stale_output in PROCESSED_DIR.glob("lsc_thematic_*"):
    if stale_output.is_file() and stale_output.name not in RETAINED_PROCESSED_OUTPUTS:
        stale_output.unlink()

LSC_FIGURE_DPI = 300
THEMATIC_NEIGHBOUR_PALETTE = ["#4F8DB3", "#C98263", "#79A889", "#A998C9", "#7B8785"]
HEATMAP_RANK_COLOURS = ["#2F6F9F", "#75A9C8", "#AECFE0", "#E1B49D", "#B66A4A"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Target Contexts

Only target contexts enter this analysis. The semantic time axis is publication year (`lsc_year`), consistent with Sentiment, Intensity, and Breadth. Non-substantive and `substantive_other` contexts remain outside the model because the thematic question concerns substantive ADHD/autism discourse.

In [2]:

if not CONTEXT_PATH.exists():
    raise FileNotFoundError(f"Missing shared LSC context table: {CONTEXT_PATH}")
if not FRAME_LABEL_PATH.exists():
    raise FileNotFoundError(f"Missing frame-label handoff: {FRAME_LABEL_PATH}")

context_columns = [
    "doc_id",
    "url",
    "registered_domain",
    "analysis_unit",
    "target_group",
    "term_role",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "target_sentence_plus_adjacent",
    "published_year",
    "source_year",
    "lsc_year",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
target_contexts = contexts.loc[contexts["analysis_unit"].isin(TARGET_UNITS)].copy()
target_contexts["registered_domain"] = target_contexts["registered_domain"].fillna("unknown_domain")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


target_contexts["context_id"] = target_contexts.apply(stable_context_id, axis=1)
frame_labels = pd.read_csv(
    FRAME_LABEL_PATH,
    usecols=["context_id", "predicted_derived_frame", "p_substantive", "p_clinical_given_substantive", "p_lived_given_substantive"],
)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context IDs.")

target_contexts = target_contexts.merge(frame_labels, on="context_id", how="left")
missing_labels = target_contexts["predicted_derived_frame"].isna().sum()
if missing_labels:
    raise RuntimeError(f"Missing frame labels for {missing_labels:,} target contexts.")

core_contexts = target_contexts.loc[target_contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)].copy()
core_contexts["base_frame_stratum"] = core_contexts["predicted_derived_frame"]

input_diagnostics = (
    target_contexts.groupby(["analysis_unit", "lsc_year", "predicted_derived_frame"], as_index=False)
    .agg(
        contexts=("context_id", "size"),
        documents=("doc_id", "nunique"),
        domains=("registered_domain", "nunique"),
    )
    .rename(columns={"predicted_derived_frame": "frame_stratum"})
)

core_contexts.groupby(["analysis_unit", "base_frame_stratum"]).size().unstack(fill_value=0)


base_frame_stratum,clinical_only,lived_only,mixed
analysis_unit,,,
ADHD,12039,3610,2000
Autism,20306,12826,6331


## Canonicalise and Tokenise

The model is trained on lemmatised content words from target-centred passages. This cell caches tokenised contexts under `data/interim/lsc/thematic_evolution/` so reruns can skip the expensive spaCy pass unless the input contexts or tokenisation settings change. Diagnostics are displayed here to check that the content filter keeps enough target evidence in every frame-year cell.


In [3]:

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
except OSError as exc:
    raise RuntimeError("spaCy model en_core_web_sm is required for thematic tokenisation.") from exc

TARGET_PATTERNS = [
    (re.compile(r"\battention\s+deficit(?:\s+hyperactivity(?:\s+disorder)?)?\b", flags=re.IGNORECASE), " adhd_concept "),
    (re.compile(r"\bADHD\b", flags=re.IGNORECASE), " adhd_concept "),
    (re.compile(r"\bautism\s+spectrum(?:\s+disorder)?\b", flags=re.IGNORECASE), " autism_concept "),
    (re.compile(r"\bASD\b", flags=re.IGNORECASE), " autism_concept "),
    (re.compile(r"\bautistic\b", flags=re.IGNORECASE), " autism_concept "),
    (re.compile(r"\bautism\b", flags=re.IGNORECASE), " autism_concept "),
]
TOKEN_RE = re.compile(r"^[a-z_]+$")
CUSTOM_STOPWORDS = {
    "http", "https", "www", "com", "org", "net", "html", "php", "pdf",
    "website", "webpage", "page", "site", "menu", "click", "login", "subscribe",
    "newsletter", "cookie", "cookies", "copyright", "reserved", "email", "facebook",
    "twitter", "instagram", "pinterest", "google", "post", "posted", "share", "search",
    "read", "more", "learn", "home", "contact", "privacy", "policy", "terms",
}
STOPWORDS = set(nlp.Defaults.stop_words) | CUSTOM_STOPWORDS


def canonicalise_text(text: object) -> str:
    canonical = str(text or "")
    for pattern, replacement in TARGET_PATTERNS:
        canonical = pattern.sub(replacement, canonical)
    return canonical


def token_from_spacy_token(token: spacy.tokens.Token) -> str | None:
    raw = token.text.lower().strip()
    if raw in TARGET_CONCEPT_TOKEN.values():
        return raw
    lemma = token.lemma_.lower().strip() if token.lemma_ else raw
    if lemma == "-pron-":
        lemma = raw
    if lemma in {"adhd", "add"}:
        return "adhd_concept"
    if lemma in {"autism", "autistic", "asd"}:
        return "autism_concept"
    if len(lemma) <= 2 or lemma in STOPWORDS:
        return None
    if not TOKEN_RE.match(lemma):
        return None
    return lemma


tokenised_contexts_path = INTERIM_DIR / "lsc_thematic_tokenised_contexts.parquet"
tokenised_manifest_path = INTERIM_DIR / "lsc_thematic_tokenised_manifest.json"


def tokenisation_manifest_for_contexts(frame: pd.DataFrame) -> dict[str, object]:
    digest = hashlib.sha256()
    for row in frame.sort_values("context_id").itertuples(index=False):
        digest.update(str(row.context_id).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.target_sentence_plus_adjacent or "").encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.base_frame_stratum).encode("utf-8"))
        digest.update(b"\0")
    return {
        "schema_version": 1,
        "tokenisation_config_version": TOKENISATION_CONFIG_VERSION,
        "contexts": int(len(frame)),
        "context_fingerprint": digest.hexdigest(),
        "min_context_tokens": MIN_CONTEXT_TOKENS,
        "spacy_model": "en_core_web_sm",
        "target_regexes": [pattern.pattern for pattern, _ in TARGET_PATTERNS],
        "custom_stopwords": sorted(CUSTOM_STOPWORDS),
        "token_pattern": TOKEN_RE.pattern,
        "target_concept_token": TARGET_CONCEPT_TOKEN,
    }


expected_tokenised_manifest = tokenisation_manifest_for_contexts(core_contexts)
token_cache_status = "created_cache"
if not REBUILD_TOKENISED_CONTEXTS and tokenised_contexts_path.exists() and tokenised_manifest_path.exists():
    observed_tokenised_manifest = json.loads(tokenised_manifest_path.read_text())
    if observed_tokenised_manifest == expected_tokenised_manifest:
        tokenised_contexts = pd.read_parquet(tokenised_contexts_path)
        token_cache_status = "loaded_cache"
    else:
        tokenised_contexts = None
else:
    tokenised_contexts = None

if tokenised_contexts is None:
    token_records = []
    texts = core_contexts["target_sentence_plus_adjacent"].fillna("").map(canonicalise_text).tolist()
    for row, doc in tqdm(
        zip(core_contexts.itertuples(index=False), nlp.pipe(texts, batch_size=256)),
        total=len(core_contexts),
        desc="Tokenising thematic contexts",
    ):
        tokens = [token for token in (token_from_spacy_token(token) for token in doc) if token]
        target_token = TARGET_CONCEPT_TOKEN[row.analysis_unit]
        token_records.append(
            {
                "context_id": row.context_id,
                "doc_id": row.doc_id,
                "registered_domain": row.registered_domain,
                "analysis_unit": row.analysis_unit,
                "lsc_year": int(row.lsc_year),
                "base_frame_stratum": row.base_frame_stratum,
                "target_concept_token": target_token,
                "tokens": tokens,
                "token_count": len(tokens),
                "target_token_count": tokens.count(target_token),
            }
        )
    tokenised_contexts = pd.DataFrame(token_records)
    tokenised_contexts["usable_context"] = tokenised_contexts["token_count"].ge(MIN_CONTEXT_TOKENS) & tokenised_contexts["target_token_count"].gt(0)
    tokenised_contexts.to_parquet(tokenised_contexts_path, index=False)
    tokenised_manifest_path.write_text(json.dumps(expected_tokenised_manifest, indent=2, sort_keys=True) + "\n")

token_diagnostics = (
    tokenised_contexts.groupby(["analysis_unit", "lsc_year", "base_frame_stratum"], as_index=False)
    .agg(
        contexts=("context_id", "size"),
        usable_contexts=("usable_context", "sum"),
        documents=("doc_id", "nunique"),
        domains=("registered_domain", "nunique"),
        total_tokens=("token_count", "sum"),
        target_token_count=("target_token_count", "sum"),
    )
)
{
    "token_cache_status": token_cache_status,
    "tokenised_context_rows": len(tokenised_contexts),
    "usable_context_rows": int(tokenised_contexts["usable_context"].sum()),
    "token_diagnostic_rows": len(token_diagnostics),
}


{'token_cache_status': 'loaded_cache',
 'tokenised_context_rows': 57112,
 'usable_context_rows': 56857,
 'token_diagnostic_rows': 78}

## Build Modelling Corpora

Each target is modelled for the substantive-core aggregate and for the two report-facing frame strata. Mixed-frame contexts contribute to the aggregate but are not plotted as a separate thematic panel.

In [4]:

usable_base = tokenised_contexts.loc[tokenised_contexts["usable_context"]].copy()

overall_rows = usable_base.copy()
overall_rows["frame_stratum"] = "substantive_core_overall"
frame_rows = usable_base.loc[usable_base["base_frame_stratum"].isin(["clinical_only", "lived_only"])].copy()
frame_rows["frame_stratum"] = frame_rows["base_frame_stratum"]
model_contexts = pd.concat([overall_rows, frame_rows], ignore_index=True, sort=False)


def count_token(sentences: list[list[str]], token: str) -> int:
    return int(sum(sentence.count(token) for sentence in sentences))


model_input_records = []
for (unit, frame), group in model_contexts.groupby(["analysis_unit", "frame_stratum"], sort=True):
    target_token = TARGET_CONCEPT_TOKEN[unit]
    sentences = group["tokens"].tolist()
    vocabulary = Counter(token for sentence in sentences for token in sentence)
    model_input_records.append(
        {
            "analysis_unit": unit,
            "frame_stratum": frame,
            "contexts": int(len(group)),
            "documents": int(group["doc_id"].nunique()),
            "domains": int(group["registered_domain"].nunique()),
            "years": int(group["lsc_year"].nunique()),
            "total_tokens": int(sum(len(sentence) for sentence in sentences)),
            "vocabulary_size_raw": int(len(vocabulary)),
            "vocabulary_size_min_count": int(sum(count >= WORD2VEC_PARAMS["min_count"] for count in vocabulary.values())),
            "target_token_count": int(vocabulary[target_token]),
        }
    )

model_input_diagnostics = pd.DataFrame(model_input_records).sort_values(["analysis_unit", "frame_stratum"])
expected_groups = pd.MultiIndex.from_product([TARGET_UNITS, MODELED_FRAME_STRATA], names=["analysis_unit", "frame_stratum"])
observed_groups = pd.MultiIndex.from_frame(model_input_diagnostics[["analysis_unit", "frame_stratum"]])
missing_groups = expected_groups.difference(observed_groups)
if len(missing_groups):
    raise RuntimeError(f"Missing thematic model groups: {list(missing_groups)}")

model_input_diagnostics


,analysis_unit,frame_stratum,contexts,documents,domains,years,total_tokens,vocabulary_size_raw,vocabulary_size_min_count,target_token_count
0,ADHD,clinical_only,11841,7889,6294,13,429055,19841,6078,22573
1,ADHD,lived_only,3590,2713,2344,13,101552,10312,2759,5455
2,ADHD,substantive_core_overall,17424,11314,8921,13,595952,24813,7703,31539
3,Autism,clinical_only,20290,12951,9310,13,776956,30156,9544,38982
4,Autism,lived_only,12818,9064,7285,13,399149,23004,6899,21585
5,Autism,substantive_core_overall,39433,24131,17009,13,1403946,43760,13991,72865


## Train Diachronic Word2Vec Models

For each target and frame stratum, the notebook trains one global skip-gram Word2Vec model and then continues training a copy of that model on each publication year. This follows the type-level embedding recipe in [Vylomova and Haslam (2021)](https://langsci-press.org/catalog/view/303/3028/2375-1): the global model supplies a shared starting space, while each year-specific model captures annual association changes.

The trained model files are cached under `data/interim/lsc/thematic_evolution/word2vec_models/`. They include the learned embeddings, so ordinary reruns load the cache and skip retraining unless the tokenised input, modelling parameters, or rebuild flag changes.

In [5]:

class TqdmEpochProgress(CallbackAny2Vec):
    def __init__(self, total: int, desc: str):
        self.progress = tqdm(total=total, desc=desc, leave=False)

    def on_epoch_end(self, model: Word2Vec) -> None:
        self.progress.update(1)
        if self.progress.n >= self.progress.total:
            self.progress.close()


def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def fingerprint_sentences(frame: pd.DataFrame) -> str:
    digest = hashlib.sha256()
    for row in frame.sort_values(["context_id", "frame_stratum"]).itertuples(index=False):
        digest.update(str(row.context_id).encode("utf-8"))
        digest.update(b"\0")
        digest.update(" ".join(row.tokens).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def group_model_paths(unit: str, frame: str) -> tuple[Path, Path, dict[int, Path], Path]:
    group_dir = MODEL_DIR / f"{slug(unit)}__{slug(frame)}"
    year_dir = group_dir / "year_models"
    year_dir.mkdir(parents=True, exist_ok=True)
    global_path = group_dir / "global_word2vec.model"
    year_paths = {year: year_dir / f"word2vec_{year}.model" for year in EXPECTED_YEARS}
    manifest_path = group_dir / "model_manifest.json"
    return group_dir, global_path, year_paths, manifest_path


def manifest_for_group(group: pd.DataFrame, unit: str, frame: str) -> dict[str, object]:
    return {
        "schema_version": 1,
        "analysis_unit": unit,
        "frame_stratum": frame,
        "expected_years": EXPECTED_YEARS,
        "word2vec_params": WORD2VEC_PARAMS,
        "context_fingerprint": fingerprint_sentences(group),
        "contexts": int(len(group)),
    }


def load_cached_models(unit: str, frame: str, expected_manifest: dict[str, object]) -> tuple[Word2Vec, dict[int, Word2Vec]] | None:
    _, global_path, year_paths, manifest_path = group_model_paths(unit, frame)
    if REBUILD_WORD2VEC_MODELS or not manifest_path.exists() or not global_path.exists():
        return None
    if any(not path.exists() for path in year_paths.values()):
        return None
    observed_manifest = json.loads(manifest_path.read_text())
    if observed_manifest != expected_manifest:
        return None
    return Word2Vec.load(str(global_path)), {year: Word2Vec.load(str(path)) for year, path in year_paths.items()}


def train_models_for_group(unit: str, frame: str, group: pd.DataFrame) -> tuple[Word2Vec, dict[int, Word2Vec], str]:
    expected_manifest = manifest_for_group(group, unit, frame)
    cached = load_cached_models(unit, frame, expected_manifest)
    if cached is not None:
        global_model, year_models = cached
        return global_model, year_models, "loaded_cache"

    sentences = group["tokens"].tolist()
    global_model = Word2Vec(
        sentences=sentences,
        vector_size=WORD2VEC_PARAMS["vector_size"],
        window=WORD2VEC_PARAMS["window"],
        min_count=WORD2VEC_PARAMS["min_count"],
        sg=WORD2VEC_PARAMS["sg"],
        workers=WORD2VEC_PARAMS["workers"],
        seed=WORD2VEC_PARAMS["seed"],
        epochs=WORD2VEC_PARAMS["global_epochs"],
        callbacks=[TqdmEpochProgress(WORD2VEC_PARAMS["global_epochs"], f"{unit} {FRAME_LABELS.get(frame, frame)} global epochs")],
    )
    target_token = TARGET_CONCEPT_TOKEN[unit]
    if target_token not in global_model.wv:
        raise RuntimeError(f"Target token {target_token} is missing from {unit}/{frame} global vocabulary.")

    year_models = {}
    for year in tqdm(EXPECTED_YEARS, desc=f"{unit} {FRAME_LABELS.get(frame, frame)} yearly models", leave=False):
        year_sentences = group.loc[group["lsc_year"].eq(year), "tokens"].tolist()
        year_model = copy.deepcopy(global_model)
        if year_sentences:
            year_model.train(
                year_sentences,
                total_examples=len(year_sentences),
                epochs=WORD2VEC_PARAMS["year_epochs"],
                callbacks=[TqdmEpochProgress(WORD2VEC_PARAMS["year_epochs"], f"{unit} {FRAME_LABELS.get(frame, frame)} {year} epochs")],
            )
        year_models[year] = year_model

    _, global_path, year_paths, manifest_path = group_model_paths(unit, frame)
    global_model.save(str(global_path))
    for year, model in year_models.items():
        model.save(str(year_paths[year]))
    manifest_path.write_text(json.dumps(expected_manifest, indent=2, sort_keys=True) + "\n")
    return global_model, year_models, "trained"


models: dict[tuple[str, str], dict[str, object]] = {}
training_records = []
model_groups = list(model_contexts.groupby(["analysis_unit", "frame_stratum"], sort=True))
for (unit, frame), group in tqdm(model_groups, desc="Training/loading thematic Word2Vec groups"):
    global_model, year_models, cache_status = train_models_for_group(unit, frame, group)
    target_token = TARGET_CONCEPT_TOKEN[unit]
    models[(unit, frame)] = {"global": global_model, "years": year_models}
    training_records.append(
        {
            "analysis_unit": unit,
            "frame_stratum": frame,
            "cache_status": cache_status,
            "contexts": int(len(group)),
            "global_corpus_count": int(global_model.corpus_count),
            "global_vocabulary_size": int(len(global_model.wv)),
            "target_in_global_vocab": bool(target_token in global_model.wv),
        }
    )

training_diagnostics = pd.DataFrame(training_records).sort_values(["analysis_unit", "frame_stratum"])
training_diagnostics


Training/loading thematic Word2Vec groups:   0%|          | 0/6 [00:00<?, ?it/s]

Training/loading thematic Word2Vec groups:  17%|█▋        | 1/6 [00:00<00:02,  2.26it/s]

Training/loading thematic Word2Vec groups:  33%|███▎      | 2/6 [00:00<00:01,  3.41it/s]

Training/loading thematic Word2Vec groups:  50%|█████     | 3/6 [00:01<00:01,  2.42it/s]

Training/loading thematic Word2Vec groups:  67%|██████▋   | 4/6 [00:01<00:01,  1.94it/s]

Training/loading thematic Word2Vec groups:  83%|████████▎ | 5/6 [00:02<00:00,  1.98it/s]

Training/loading thematic Word2Vec groups: 100%|██████████| 6/6 [00:03<00:00,  1.40it/s]

Training/loading thematic Word2Vec groups: 100%|██████████| 6/6 [00:03<00:00,  1.73it/s]

,analysis_unit,frame_stratum,cache_status,contexts,global_corpus_count,global_vocabulary_size,target_in_global_vocab
0,ADHD,clinical_only,loaded_cache,11841,11841,6078,True
1,ADHD,lived_only,loaded_cache,3590,3590,2759,True
2,ADHD,substantive_core_overall,loaded_cache,17424,17424,7703,True
3,Autism,clinical_only,loaded_cache,20290,20290,9544,True
4,Autism,lived_only,loaded_cache,12818,12818,6899,True
5,Autism,substantive_core_overall,loaded_cache,39433,39433,13991,True


## Annual Neighbours and Stable Similarity Trajectories

The annual top-neighbour table keeps the full top-five list for every target, frame, and year. This table is useful as an audit trail and for appendix heatmaps, so it may contain one-off neighbours that are not suitable for a compact line figure. The report-facing line figures therefore use a moderate stability rule: plotted neighbours must appear in annual top-five lists in at least two years and have finite cosine-similarity values in at least ten years. Panels may show fewer than five lines.


In [6]:

def yearly_token_counts(group: pd.DataFrame) -> dict[int, Counter[str]]:
    counts = {}
    for year, year_group in group.groupby("lsc_year"):
        counts[int(year)] = Counter(token for sentence in year_group["tokens"] for token in sentence)
    return counts


def top_neighbours_for_year(model: Word2Vec, target_token: str, counts: Counter[str]) -> list[tuple[int, str, float, int]]:
    candidates = [
        word
        for word, count in counts.items()
        if count >= MIN_NEIGHBOUR_YEAR_TOKEN_COUNT and word != target_token and word in model.wv
    ]
    similarities = [(word, float(model.wv.similarity(target_token, word)), int(counts[word])) for word in candidates]
    similarities.sort(key=lambda item: (-item[1], item[0]))
    return [(rank, word, similarity, count) for rank, (word, similarity, count) in enumerate(similarities[:ANNUAL_TOP_N], start=1)]


def similarity_trajectory_for_candidate(
    unit: str,
    frame: str,
    group: pd.DataFrame,
    neighbour: str,
    counts_by_year: dict[int, Counter[str]],
) -> list[dict[str, object]]:
    target_token = TARGET_CONCEPT_TOKEN[unit]
    top_subset = annual_neighbours.loc[annual_neighbours["analysis_unit"].eq(unit) & annual_neighbours["frame_stratum"].eq(frame)]
    records = []
    for year in EXPECTED_YEARS:
        counts = counts_by_year.get(year, Counter())
        target_count = int(counts[target_token])
        neighbour_count = int(counts[neighbour])
        model = models[(unit, frame)]["years"][year]
        similarity = np.nan
        if target_count >= MIN_TARGET_YEAR_TOKEN_COUNT and neighbour_count >= MIN_NEIGHBOUR_YEAR_TOKEN_COUNT and neighbour in model.wv:
            similarity = float(model.wv.similarity(target_token, neighbour))
        rank_row = top_subset.loc[top_subset["lsc_year"].eq(year) & top_subset["neighbour"].eq(neighbour)]
        records.append(
            {
                "analysis_unit": unit,
                "frame_stratum": frame,
                "lsc_year": year,
                "neighbour": neighbour,
                "cosine_similarity": similarity,
                "target_token": target_token,
                "target_token_count": target_count,
                "neighbour_token_count": neighbour_count,
                "annual_top5_rank": int(rank_row.iloc[0]["rank"]) if not rank_row.empty else np.nan,
            }
        )
    return records


annual_records = []
for (unit, frame), group in tqdm(model_groups, desc="Extracting annual top neighbours"):
    target_token = TARGET_CONCEPT_TOKEN[unit]
    counts_by_year = yearly_token_counts(group)
    for year in EXPECTED_YEARS:
        year_group = group.loc[group["lsc_year"].eq(year)]
        counts = counts_by_year.get(year, Counter())
        target_count = int(counts[target_token])
        neighbours = top_neighbours_for_year(models[(unit, frame)]["years"][year], target_token, counts) if target_count >= MIN_TARGET_YEAR_TOKEN_COUNT else []
        for rank, word, similarity, neighbour_count in neighbours:
            annual_records.append(
                {
                    "analysis_unit": unit,
                    "frame_stratum": frame,
                    "lsc_year": year,
                    "rank": rank,
                    "neighbour": word,
                    "cosine_similarity": similarity,
                    "target_token": target_token,
                    "target_token_count": target_count,
                    "neighbour_token_count": neighbour_count,
                    "contexts": int(len(year_group)),
                    "documents": int(year_group["doc_id"].nunique()),
                    "domains": int(year_group["registered_domain"].nunique()),
                }
            )

annual_neighbours = pd.DataFrame(annual_records).sort_values(["analysis_unit", "frame_stratum", "lsc_year", "rank"])
annual_neighbours_path = PROCESSED_DIR / "lsc_thematic_annual_top_neighbours.csv"
annual_neighbours.to_csv(annual_neighbours_path, index=False)

selection_records = []
trajectory_records = []
for (unit, frame), group in tqdm(model_groups, desc="Selecting stable neighbour trajectories"):
    counts_by_year = yearly_token_counts(group)
    top_subset = annual_neighbours.loc[annual_neighbours["analysis_unit"].eq(unit) & annual_neighbours["frame_stratum"].eq(frame)]
    candidates = (
        top_subset.groupby("neighbour", as_index=False)
        .agg(
            years_seen_in_top5=("lsc_year", "nunique"),
            mean_top_similarity=("cosine_similarity", "mean"),
            best_rank=("rank", "min"),
            first_top_year=("lsc_year", "min"),
            last_top_year=("lsc_year", "max"),
        )
    )
    candidates = candidates.loc[candidates["years_seen_in_top5"].ge(MIN_PLOTTED_TOP5_YEARS)].copy()

    candidate_records = []
    for candidate in candidates.itertuples(index=False):
        records = similarity_trajectory_for_candidate(unit, frame, group, candidate.neighbour, counts_by_year)
        finite_years = int(pd.Series([record["cosine_similarity"] for record in records]).notna().sum())
        if finite_years < MIN_PLOTTED_FINITE_YEARS:
            continue
        candidate_records.append((candidate, records, finite_years))

    candidate_records.sort(key=lambda item: (-item[0].years_seen_in_top5, -item[0].mean_top_similarity, item[0].best_rank, item[0].neighbour))
    for plot_order, (candidate, records, finite_years) in enumerate(candidate_records[:PLOTTED_NEIGHBOURS], start=1):
        selection_records.append(
            {
                "analysis_unit": unit,
                "frame_stratum": frame,
                "plot_order": plot_order,
                "neighbour": candidate.neighbour,
                "years_seen_in_top5": int(candidate.years_seen_in_top5),
                "finite_similarity_years": finite_years,
                "mean_top_similarity": float(candidate.mean_top_similarity),
                "best_rank": int(candidate.best_rank),
                "first_top_year": int(candidate.first_top_year),
                "last_top_year": int(candidate.last_top_year),
            }
        )
        for record in records:
            record["plot_order"] = plot_order
            trajectory_records.append(record)

plotted_neighbours = pd.DataFrame(selection_records).sort_values(["analysis_unit", "frame_stratum", "plot_order"])
trajectories = pd.DataFrame(trajectory_records).sort_values(["analysis_unit", "frame_stratum", "plot_order", "lsc_year"])
plotted_neighbours_path = PROCESSED_DIR / "lsc_thematic_plotted_neighbours.csv"
trajectory_path = PROCESSED_DIR / "lsc_thematic_neighbour_similarity_trajectories.csv"
plotted_neighbours.to_csv(plotted_neighbours_path, index=False)
trajectories.to_csv(trajectory_path, index=False)

plotted_neighbours


Extracting annual top neighbours:   0%|          | 0/6 [00:00<?, ?it/s]

Extracting annual top neighbours:  17%|█▋        | 1/6 [00:00<00:00,  6.20it/s]

Extracting annual top neighbours:  50%|█████     | 3/6 [00:00<00:00,  7.18it/s]

Extracting annual top neighbours:  67%|██████▋   | 4/6 [00:00<00:00,  5.49it/s]

Extracting annual top neighbours:  83%|████████▎ | 5/6 [00:00<00:00,  5.71it/s]

...

Selecting stable neighbour trajectories:   0%|          | 0/6 [00:00<?, ?it/s]

Selecting stable neighbour trajectories:  50%|█████     | 3/6 [00:00<00:00, 16.42it/s]

Selecting stable neighbour trajectories:  83%|████████▎ | 5/6 [00:00<00:00, 14.60it/s]

Selecting stable neighbour trajectories: 100%|██████████| 6/6 [00:00<00:00, 12.57it/s]

,analysis_unit,frame_stratum,plot_order,neighbour,years_seen_in_top5,finite_similarity_years,mean_top_similarity,best_rank,first_top_year,last_top_year
0,ADHD,clinical_only,1,disorder,13,13,0.520584,1,2014,2026
1,ADHD,clinical_only,2,child,12,13,0.471077,1,2014,2026
2,ADHD,clinical_only,3,condition,7,13,0.434217,2,2014,2025
3,ADHD,clinical_only,4,symptom,7,13,0.415577,2,2017,2025
4,ADHD,clinical_only,5,diagnose,3,13,0.423621,3,2015,2023
5,ADHD,lived_only,1,help,4,13,0.493003,1,2014,2026
6,ADHD,lived_only,2,child,2,13,0.528454,1,2016,2020
7,ADHD,lived_only,3,diagnose,2,13,0.499815,3,2015,2022
8,ADHD,substantive_core_overall,1,child,12,13,0.517287,1,2014,2026
9,ADHD,substantive_core_overall,2,disorder,12,13,0.516719,1,2014,2026


## Descriptive Neighbour Trends

The slopes below are only reading aids for the selected neighbour trajectories. They are not exported as a separate processed result because thematic evolution is treated as an interpretive layer rather than as another scalar regression table.


In [7]:

trend_records = []
for (unit, frame, neighbour), series in trajectories.groupby(["analysis_unit", "frame_stratum", "neighbour"], sort=True):
    data = series[["lsc_year", "cosine_similarity"]].dropna().sort_values("lsc_year")
    if len(data) >= 3 and data["cosine_similarity"].nunique() > 1:
        x = data["lsc_year"].to_numpy(dtype=float) - data["lsc_year"].mean()
        y = data["cosine_similarity"].to_numpy(dtype=float)
        result = stats.linregress(x, y)
        start_value = float(data.loc[data["lsc_year"].idxmin(), "cosine_similarity"])
        end_value = float(data.loc[data["lsc_year"].idxmax(), "cosine_similarity"])
        trend_records.append(
            {
                "analysis_unit": unit,
                "frame_stratum": frame,
                "neighbour": neighbour,
                "n_years": int(len(data)),
                "first_year": int(data["lsc_year"].min()),
                "last_year": int(data["lsc_year"].max()),
                "first_similarity": start_value,
                "last_similarity": end_value,
                "end_minus_start": end_value - start_value,
                "linear_slope_per_year": float(result.slope),
                "linear_p_value_descriptive": float(result.pvalue),
            }
        )

trend_summary = pd.DataFrame(trend_records).sort_values(["analysis_unit", "frame_stratum", "linear_slope_per_year"])
trend_summary.head(12)


,analysis_unit,frame_stratum,neighbour,n_years,first_year,last_year,first_similarity,last_similarity,end_minus_start,linear_slope_per_year,linear_p_value_descriptive
0,ADHD,clinical_only,child,13,2014,2026,0.494118,0.421330,-0.072788,-0.007139,0.071561
3,ADHD,clinical_only,disorder,13,2014,2026,0.560175,0.547976,-0.012199,-0.003987,0.176704
2,ADHD,clinical_only,diagnose,13,2014,2026,0.440377,0.400974,-0.039402,-0.000140,0.969393
1,ADHD,clinical_only,condition,13,2014,2026,0.479163,0.413937,-0.065226,0.003586,0.254256
4,ADHD,clinical_only,symptom,13,2014,2026,0.324944,0.315238,-0.009706,0.005292,0.100154
5,ADHD,lived_only,child,13,2014,2026,0.365890,0.326138,-0.039752,-0.012668,0.064826
6,ADHD,lived_only,diagnose,13,2014,2026,0.339830,0.266874,-0.072956,-0.011901,0.118902
7,ADHD,lived_only,help,13,2014,2026,0.485247,0.493742,0.008495,0.004402,0.374414
11,ADHD,substantive_core_overall,disorder,13,2014,2026,0.551420,0.416586,-0.134834,-0.014635,0.001601
8,ADHD,substantive_core_overall,child,13,2014,2026,0.537846,0.481911,-0.055936,-0.010781,0.001683


## Report Figures

The main figures show stable neighbour trajectories for ADHD and Autism. The appendix heatmaps are exploratory diagnostics: they show annual top-five churn and make it clear which neighbours are persistent versus one-year entries. 2026 is retained for consistency with the other LSC analyses, but the compact context diagnostics in the handoff cell should be checked before interpreting one-year neighbour changes.


In [8]:

def style_axis(ax: plt.Axes) -> None:
    ax.grid(axis="y", color="#D8DEE3", linewidth=0.7, alpha=0.8)
    ax.grid(axis="x", color="#EDF0F2", linewidth=0.5, alpha=0.8)
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(labelsize=8.5)


def plot_unit_figure(unit: str) -> tuple[Path, Path]:
    unit_data = trajectories.loc[trajectories["analysis_unit"].eq(unit)].copy()
    finite_values = unit_data["cosine_similarity"].replace([np.inf, -np.inf], np.nan).dropna()
    if finite_values.empty:
        raise RuntimeError(f"No finite neighbour similarities available for {unit}.")
    y_padding = max((finite_values.max() - finite_values.min()) * 0.12, 0.025)
    y_low, y_high = finite_values.min() - y_padding, finite_values.max() + y_padding

    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.4), sharex=True, sharey=True)
    for ax, frame in zip(axes, MODELED_FRAME_STRATA):
        panel = unit_data.loc[unit_data["frame_stratum"].eq(frame)]
        if panel.empty:
            ax.text(0.5, 0.5, "No stable neighbours", ha="center", va="center", transform=ax.transAxes, fontsize=9, color="#6B7280")
        for idx, (neighbour, series) in enumerate(panel.groupby("neighbour", sort=False)):
            series = series.sort_values("lsc_year")
            color = THEMATIC_NEIGHBOUR_PALETTE[idx % len(THEMATIC_NEIGHBOUR_PALETTE)]
            ax.plot(
                series["lsc_year"],
                series["cosine_similarity"],
                marker="o",
                markersize=3.8,
                linewidth=1.8,
                color=color,
                label=neighbour.replace("_", " "),
            )
        ax.set_title(FRAME_LABELS[frame], fontsize=11, fontweight="bold", color="#263238", pad=9)
        ax.set_ylim(y_low, y_high)
        ax.set_xlabel("Publication year", fontsize=9.5)
        style_axis(ax)
        if not panel.empty:
            ax.legend(frameon=False, fontsize=7.2, loc="best", handlelength=1.8)
    axes[0].set_ylabel("Cosine similarity to target concept", fontsize=9.5)
    fig.suptitle(f"Thematic neighbours of {unit} discourse", fontsize=14, fontweight="bold", x=0.02, y=0.985, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.955], pad=1.0)
    png_path = FIGURE_DIR / f"lsc_thematic_neighbour_similarity_{slug(unit)}.png"
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return png_path, pdf_path


def sorted_heatmap_neighbours(panel: pd.DataFrame) -> list[str]:
    order = (
        panel.groupby("neighbour", as_index=False)
        .agg(
            years_seen=("lsc_year", "nunique"),
            mean_rank=("rank", "mean"),
            first_year=("lsc_year", "min"),
            best_rank=("rank", "min"),
        )
        .sort_values(["years_seen", "mean_rank", "best_rank", "first_year", "neighbour"], ascending=[False, True, True, True, True])
    )
    return order["neighbour"].tolist()


def plot_unit_heatmap(unit: str) -> tuple[Path, Path]:
    unit_rows = annual_neighbours.loc[annual_neighbours["analysis_unit"].eq(unit)].copy()
    row_counts = [max(1, unit_rows.loc[unit_rows["frame_stratum"].eq(frame), "neighbour"].nunique()) for frame in MODELED_FRAME_STRATA]
    fig_height = max(6.0, 1.0 + 0.18 * sum(row_counts))
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(10.8, fig_height),
        sharex=True,
        gridspec_kw={"height_ratios": row_counts},
    )
    cmap = mcolors.ListedColormap(HEATMAP_RANK_COLOURS)
    cmap.set_bad("#FFFFFF")
    norm = mcolors.BoundaryNorm(np.arange(0.5, ANNUAL_TOP_N + 1.5, 1), cmap.N)

    image = None
    for ax, frame in zip(axes, MODELED_FRAME_STRATA):
        panel = unit_rows.loc[unit_rows["frame_stratum"].eq(frame)]
        neighbours = sorted_heatmap_neighbours(panel)
        matrix = np.full((len(neighbours), len(EXPECTED_YEARS)), np.nan)
        row_index = {neighbour: idx for idx, neighbour in enumerate(neighbours)}
        col_index = {year: idx for idx, year in enumerate(EXPECTED_YEARS)}
        for row in panel.itertuples(index=False):
            matrix[row_index[row.neighbour], col_index[int(row.lsc_year)]] = row.rank
        image = ax.imshow(np.ma.masked_invalid(matrix), aspect="auto", cmap=cmap, norm=norm)
        ax.set_title(FRAME_LABELS[frame], fontsize=11, fontweight="bold", color="#263238", pad=8)
        ax.set_yticks(range(len(neighbours)))
        ax.set_yticklabels([neighbour.replace("_", " ") for neighbour in neighbours], fontsize=7.3)
        ax.set_xticks(range(len(EXPECTED_YEARS)))
        ax.set_xticklabels(EXPECTED_YEARS, rotation=0, fontsize=8)
        ax.tick_params(axis="y", length=0)
        ax.set_xticks(np.arange(-0.5, len(EXPECTED_YEARS), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(neighbours), 1), minor=True)
        ax.grid(which="minor", color="#EDF0F2", linewidth=0.6)
        ax.tick_params(which="minor", bottom=False, left=False)
    axes[-1].set_xlabel("Publication year", fontsize=9.5)
    fig.suptitle(f"Annual top-neighbour churn for {unit} discourse", fontsize=14, fontweight="bold", x=0.02, y=0.995, ha="left")
    fig.text(0.02, 0.975, "Exploratory appendix diagnostic: coloured cells mark annual top-5 rank; blank cells are outside the annual top-5.", fontsize=9, color="#4B5563", ha="left")
    fig.tight_layout(rect=[0, 0, 0.90, 0.955], h_pad=1.5)
    if image is not None:
        cbar_ax = fig.add_axes([0.925, 0.36, 0.012, 0.28])
        cbar = fig.colorbar(image, cax=cbar_ax, orientation="vertical", ticks=range(1, ANNUAL_TOP_N + 1))
        cbar.ax.invert_yaxis()
        cbar.set_label("Top-5 rank", fontsize=8.5)
        cbar.ax.tick_params(labelsize=8)
    png_path = APPENDIX_FIGURE_DIR / f"lsc_thematic_neighbour_rank_heatmap_{slug(unit)}.png"
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return png_path, pdf_path


figure_paths = []
for unit in TARGET_UNITS:
    figure_paths.extend(plot_unit_figure(unit))
    figure_paths.extend(plot_unit_heatmap(unit))

figure_paths


[PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_adhd.png'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_adhd.pdf'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/appendix/lsc_thematic_neighbour_rank_heatmap_adhd.png'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/appendix/lsc_thematic_neighbour_rank_heatmap_adhd.pdf'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_autism.png'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_autism.pdf'),
 PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-s

## Notebook Diagnostics and Handoff

This cell displays the diagnostics needed to interpret the run without exporting a large collection of intermediate CSVs. The Overall/Mixed diagnostic is intentionally notebook-only: it helps explain whether an Overall neighbour is shared across clinical, lived, and mixed contexts or disproportionately associated with one base frame.


In [9]:

def context_token_count(tokens: object, word: str) -> int:
    return list(tokens).count(word)


model_year_diagnostics = (
    model_contexts.groupby(["analysis_unit", "frame_stratum", "lsc_year"], as_index=False)
    .agg(
        contexts=("context_id", "size"),
        documents=("doc_id", "nunique"),
        domains=("registered_domain", "nunique"),
        target_token_count=("target_token_count", "sum"),
    )
)
context_range_diagnostics = (
    model_year_diagnostics.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        min_contexts=("contexts", "min"),
        max_contexts=("contexts", "max"),
        contexts_2026=("contexts", lambda values: int(model_year_diagnostics.loc[values.index[model_year_diagnostics.loc[values.index, "lsc_year"].eq(2026)], "contexts"].iloc[0])),
        min_target_tokens=("target_token_count", "min"),
        target_tokens_2026=("target_token_count", lambda values: int(model_year_diagnostics.loc[values.index[model_year_diagnostics.loc[values.index, "lsc_year"].eq(2026)], "target_token_count"].iloc[0])),
    )
)

overall_selected = plotted_neighbours.loc[plotted_neighbours["frame_stratum"].eq("substantive_core_overall")].copy()
frame_contribution_records = []
for row in overall_selected.itertuples(index=False):
    unit_base = usable_base.loc[usable_base["analysis_unit"].eq(row.analysis_unit)]
    for frame, group in unit_base.groupby("base_frame_stratum", sort=True):
        neighbour_counts = group["tokens"].map(lambda tokens: context_token_count(tokens, row.neighbour))
        target_tokens = int(group["target_token_count"].sum())
        frame_contribution_records.append(
            {
                "analysis_unit": row.analysis_unit,
                "overall_neighbour": row.neighbour,
                "base_frame_stratum": frame,
                "contexts": int(len(group)),
                "contexts_with_neighbour": int((neighbour_counts > 0).sum()),
                "neighbour_tokens": int(neighbour_counts.sum()),
                "target_token_count": target_tokens,
                "contexts_with_neighbour_share": float((neighbour_counts > 0).sum() / len(group)) if len(group) else np.nan,
                "neighbour_tokens_per_1000_target_tokens": float(neighbour_counts.sum() / target_tokens * 1000) if target_tokens else np.nan,
            }
        )

overall_frame_contribution = pd.DataFrame(frame_contribution_records).sort_values(["analysis_unit", "overall_neighbour", "base_frame_stratum"])

audit_records = []
for (unit, frame), group in model_contexts.groupby(["analysis_unit", "frame_stratum"], sort=True):
    target_token = TARGET_CONCEPT_TOKEN[unit]
    counts_by_year = yearly_token_counts(group)
    for year in EXPECTED_YEARS:
        counts = counts_by_year.get(year, Counter())
        target_count = int(counts[target_token])
        top_rows = annual_neighbours.loc[
            annual_neighbours["analysis_unit"].eq(unit)
            & annual_neighbours["frame_stratum"].eq(frame)
            & annual_neighbours["lsc_year"].eq(year)
        ]
        if target_count < MIN_TARGET_YEAR_TOKEN_COUNT:
            audit_records.append(
                {"analysis_unit": unit, "frame_stratum": frame, "lsc_year": year, "check": "low_target_token_count", "value": target_count}
            )
        if len(top_rows) < ANNUAL_TOP_N:
            audit_records.append(
                {"analysis_unit": unit, "frame_stratum": frame, "lsc_year": year, "check": "fewer_than_top5_neighbours", "value": int(len(top_rows))}
            )

for (unit, frame, neighbour), series in trajectories.groupby(["analysis_unit", "frame_stratum", "neighbour"], sort=True):
    finite_years = int(series["cosine_similarity"].notna().sum())
    if finite_years < MIN_PLOTTED_FINITE_YEARS:
        audit_records.append(
            {"analysis_unit": unit, "frame_stratum": frame, "lsc_year": np.nan, "check": f"few_finite_years_for_{neighbour}", "value": finite_years}
        )

audit_flags = pd.DataFrame(audit_records, columns=["analysis_unit", "frame_stratum", "lsc_year", "check", "value"])

summary = {
    "token_cache_status": token_cache_status,
    "tokenised_context_rows": int(len(tokenised_contexts)),
    "usable_base_contexts": int(len(usable_base)),
    "model_context_rows": int(len(model_contexts)),
    "trained_or_loaded_model_groups": int(len(training_diagnostics)),
    "annual_top_neighbour_rows": int(len(annual_neighbours)),
    "plotted_neighbour_rows": int(len(plotted_neighbours)),
    "trajectory_rows": int(len(trajectories)),
    "audit_flag_rows": int(len(audit_flags)),
    "retained_processed_outputs": sorted(RETAINED_PROCESSED_OUTPUTS),
    "figures": [str(path.relative_to(PROJECT_ROOT)) for path in figure_paths],
}
summary_path = PROCESSED_DIR / "lsc_thematic_execution_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")

if len(training_diagnostics) != len(TARGET_UNITS) * len(MODELED_FRAME_STRATA):
    raise RuntimeError("Unexpected number of thematic model groups.")
if annual_neighbours.empty:
    raise RuntimeError("No annual thematic neighbours were produced.")
if trajectories.empty:
    raise RuntimeError("No thematic neighbour trajectories were produced.")

{
    "summary": summary,
    "context_range_diagnostics": context_range_diagnostics,
    "overall_frame_contribution": overall_frame_contribution,
    "audit_flags": audit_flags,
}


{'summary': {'token_cache_status': 'loaded_cache',
  'tokenised_context_rows': 57112,
  'usable_base_contexts': 56857,
  'model_context_rows': 105396,
  'trained_or_loaded_model_groups': 6,
  'annual_top_neighbour_rows': 390,
  'plotted_neighbour_rows': 28,
  'trajectory_rows': 364,
  'audit_flag_rows': 0,
  'retained_processed_outputs': ['lsc_thematic_annual_top_neighbours.csv',
   'lsc_thematic_execution_summary.json',
   'lsc_thematic_neighbour_similarity_trajectories.csv',
   'lsc_thematic_plotted_neighbours.csv'],
  'figures': ['reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_adhd.png',
   'reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_adhd.pdf',
   'reports/figures/lsc/thematic_evolution/appendix/lsc_thematic_neighbour_rank_heatmap_adhd.png',
   'reports/figures/lsc/thematic_evolution/appendix/lsc_thematic_neighbour_rank_heatmap_adhd.pdf',
   'reports/figures/lsc/thematic_evolution/lsc_thematic_neighbour_similarity_autism.png',
